# Module 7 — Batch query the deployed compound agent

We've built the agent; now we'll exercise it the way a downstream
analyst pipeline would: a Delta table of questions in, a Delta table of
answers out. We use `ai_query` so this scales horizontally — no Python
loop, no rate-limiting headaches, just SQL that fans questions across
the serving endpoint.

Questions cover both cyber threat-intel **and** network operations
topics — the agent picks the right tools (Genie / Knowledge Assistant /
CVE lookup / netops outages) per question.

In [ ]:
%run ./_config


In [ ]:
# Wait for the agent endpoint to be Ready before sending traffic
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()
for attempt in range(60):
    ep = w.serving_endpoints.get(AGENT_ENDPOINT)
    state = (ep.state.ready.value if ep.state and ep.state.ready else 'UNKNOWN')
    cfg = (ep.state.config_update.value if ep.state and ep.state.config_update else 'NONE')
    print(f"  [{attempt:2}] ready={state} config_update={cfg}")
    if state == 'READY':
        break
    time.sleep(15)
else:
    raise RuntimeError(f'Endpoint {AGENT_ENDPOINT} did not become READY in 15 minutes')
print(f'\nEndpoint {AGENT_ENDPOINT} is ready.')


## Build the question set

Mix of cyber threat-intel and network ops. The agent will route each to
the right tool: structured questions go to Genie, document/policy questions
go to the Knowledge Assistant, outage questions go to `netops_outages_query`.

In [ ]:
# Just one question for the dry-run — exercises the full agent multi-tool path.
# Add more rows to scale.
questions = [
    ('cross', 'Are any of the recent network outages potentially correlated with active CVEs in the KEV catalog? Use Genie for KEV stats and netops_outages_query for outage data, then synthesize.'),
]

from pyspark.sql import Row
df_q = spark.createDataFrame([Row(category=c, question=q) for c,q in questions])
df_q.write.mode('overwrite').saveAsTable(f'{BASE}.agent_questions')
df_q.display()


## Fan questions across the agent endpoint with `ai_query`

`ai_query` calls the serving endpoint per row in parallel. The agent is a
ResponsesAgent, so we pass a chat-style request and unwrap the response.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {BASE}.agent_answers_raw AS
SELECT
  category,
  question,
  ai_query(
    '{AGENT_ENDPOINT}',
    to_json(named_struct(
      'input', array(named_struct('role', 'user', 'content', question))
    )),
    failOnError => false
  ) AS response
FROM {BASE}.agent_questions
""")

# Walk the ResponsesAgent payload on the driver. Surface:
#  - the final assistant text
#  - every tool/function call the agent made along the way
#  - the total number of steps
import json
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

def summarize(value, n=300):
    if value is None: return None
    s = value if isinstance(value, str) else json.dumps(value, default=str)
    return s if len(s) <= n else s[:n] + '...'

rows = spark.table(f'{BASE}.agent_answers_raw').collect()
out = []
for r in rows:
    err = r.response.errorMessage
    raw = r.response.result
    answer = None
    tool_calls = []
    step_count = 0
    if raw:
        try:
            d = json.loads(raw)
            output = d.get('output') if isinstance(d.get('output'), list) else []
            step_count = len(output)
            # walk every output item; collect tool/function calls and their outputs
            pending = {}
            for item in output:
                t = item.get('type', '')
                if t in ('function_call', 'tool_call'):
                    cid = item.get('call_id') or item.get('id') or f'step{len(tool_calls)}'
                    pending[cid] = {
                        'tool': item.get('name') or item.get('function', {}).get('name'),
                        'arguments': summarize(item.get('arguments') or item.get('function', {}).get('arguments')),
                        'output_summary': None,
                    }
                elif t in ('function_call_output', 'tool_call_output', 'tool_result'):
                    cid = item.get('call_id') or item.get('id')
                    out_val = item.get('output') or item.get('content')
                    if cid in pending:
                        pending[cid]['output_summary'] = summarize(out_val)
                    else:
                        tool_calls.append({'tool': '(orphan_output)', 'arguments': None, 'output_summary': summarize(out_val)})
            tool_calls.extend(pending.values())
            # final assistant message
            for item in reversed(output):
                content = item.get('content') or []
                if isinstance(content, list):
                    for piece in content:
                        if isinstance(piece, dict) and 'text' in piece:
                            answer = piece['text']; break
                if answer: break
            if not answer and isinstance(d.get('choices'), list) and d['choices']:
                answer = d['choices'][0].get('message', {}).get('content')
        except Exception as e:
            answer = f'(parse error: {e})'
    out.append(Row(
        category=r.category,
        question=r.question,
        error=err,
        step_count=step_count,
        tool_calls=[Row(**tc) for tc in tool_calls],
        answer_text=answer or '(no answer)',
    ))

schema = StructType([
    StructField('category', StringType()),
    StructField('question', StringType()),
    StructField('error', StringType()),
    StructField('step_count', IntegerType()),
    StructField('tool_calls', ArrayType(StructType([
        StructField('tool', StringType()),
        StructField('arguments', StringType()),
        StructField('output_summary', StringType()),
    ]))),
    StructField('answer_text', StringType()),
])
spark.sql(f'DROP TABLE IF EXISTS {BASE}.agent_answers')
spark.createDataFrame(out, schema=schema).write.mode('overwrite').saveAsTable(f'{BASE}.agent_answers')
spark.table(f'{BASE}.agent_answers').select('category','question','step_count','tool_calls','answer_text').display()


## Sanity: count answers per category and flag failures

In [ ]:
spark.sql(f"""
SELECT
  category,
  COUNT(*) AS asked,
  SUM(CASE WHEN answer_text IS NULL OR length(answer_text) < 20 THEN 1 ELSE 0 END) AS failed_or_empty
FROM {BASE}.agent_answers
GROUP BY category
ORDER BY category
""").display()
